In [ ]:
# Simple RAG with AWS OpenSearch & Bedrock

## 📖 Overview

This notebook demonstrates the **simplest form of Retrieval-Augmented Generation (RAG)** using AWS managed services:
- **AWS OpenSearch Serverless** for vector storage and retrieval
- **Amazon Bedrock** for embeddings (Titan) and generation (Claude)

### What is Simple RAG?

Simple RAG is the foundational pattern for combining retrieval with generation:
1. **Index**: Documents are split into chunks, embedded, and stored in a vector database
2. **Retrieve**: User query is embedded and similar documents are retrieved
3. **Generate**: Retrieved documents provide context for the LLM to generate an answer

### When to Use

✅ **Good for:**
- Getting started with RAG
- Simple Q&A over documents
- Baseline for comparison
- Small to medium document collections

❌ **Not ideal for:**
- Complex multi-hop reasoning
- When precision is critical (consider reranking)
- Very large document collections (consider hierarchical indices)
- Domain-specific retrieval (consider fine-tuned embeddings)

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Generate Query Embedding<br/>Bedrock Titan]
    B --> C[Vector Search<br/>OpenSearch]
    C --> D[Retrieve Top-K Documents]
    D --> E[Format Context]
    E --> F[Generate Answer<br/>Claude Opus]
    F --> G[Return Answer]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#fff9c4
    style F fill:#ffe0b2
    style G fill:#c8e6c9
```

## 1️⃣ Setup & Imports

First, let's import the necessary libraries and our AWS utility modules.

In [ ]:
# Standard libraries
import sys
import json
from typing import List, Dict
import time

# Add parent directory to path for imports
sys.path.append('..')

# AWS utility modules
from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM, BedrockRAG
from aws_utils.rag_evaluator import RAGEvaluator
from aws_utils.diagram_generator import generate_simple_rag_diagram

# Document processing
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader

print("✓ Imports successful")

## 2️⃣ Configuration

Configure AWS services and RAG parameters.

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'simple_rag_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'  # 1024 dimensions
LLM_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Claude Opus

# RAG Parameters
CHUNK_SIZE = 1000  # Characters per chunk
CHUNK_OVERLAP = 200  # Overlap between chunks
TOP_K = 5  # Number of documents to retrieve

print(f"Configuration:")
print(f"  Region: {AWS_REGION}")
print(f"  Index: {INDEX_NAME}")
print(f"  Embedding Model: {EMBEDDING_MODEL}")
print(f"  LLM Model: {LLM_MODEL}")
print(f"  Chunk Size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}")
print(f"  Top-K: {TOP_K}")

## 3️⃣ Initialize AWS Services

Connect to OpenSearch and initialize Bedrock clients.

In [ ]:
# Initialize OpenSearch Manager
print("Connecting to OpenSearch Serverless...")
opensearch = OpenSearchManager(
    region=AWS_REGION,
    index_name=INDEX_NAME
)
print("✓ Connected to OpenSearch")

# Initialize Bedrock Embeddings
print("\nInitializing Bedrock Embeddings...")
embedder = BedrockEmbeddings(
    region_name=AWS_REGION,
    model_id=EMBEDDING_MODEL
)
print(f"✓ Embeddings ready (dimension: {embedder.get_embedding_dimension()})")

# Initialize Bedrock LLM
print("\nInitializing Bedrock LLM...")
llm = BedrockLLM(
    region_name=AWS_REGION,
    model_id=LLM_MODEL,
    temperature=0.7,
    max_tokens=2000
)
print("✓ LLM ready")

# Initialize complete RAG system
print("\nInitializing RAG system...")
rag_system = BedrockRAG(
    opensearch_manager=opensearch,
    embedding_model=EMBEDDING_MODEL,
    llm_model=LLM_MODEL,
    region=AWS_REGION
)
print("✓ RAG system ready")

## 4️⃣ Load & Process Documents

Load sample documents and split them into chunks for indexing.

### Chunking Strategy

We use `RecursiveCharacterTextSplitter` which:
- Tries to keep paragraphs together
- Falls back to sentences if needed
- Creates overlapping chunks to maintain context

**Key Parameters:**
- `chunk_size`: Maximum characters per chunk (1000)
- `chunk_overlap`: Characters shared between chunks (200)
- Higher overlap = better context preservation but more storage

In [ ]:
# Sample documents (replace with your data)
sample_documents = [
    "Amazon Web Services (AWS) is a comprehensive cloud computing platform. It offers over 200 services including compute, storage, databases, analytics, machine learning, and more.",
    "AWS Bedrock provides access to foundation models from leading AI companies through a single API. You can use models from Anthropic (Claude), Amazon (Titan), and others.",
    "OpenSearch is a distributed search and analytics engine. OpenSearch Serverless removes infrastructure management, automatically scaling compute and storage.",
    "Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. It retrieves relevant documents and uses them as context for generating accurate, grounded responses.",
    "Vector embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings, enabling semantic search."
]

print(f"Loaded {len(sample_documents)} documents")

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split documents
chunks = []
for i, doc in enumerate(sample_documents):
    doc_chunks = text_splitter.split_text(doc)
    for j, chunk in enumerate(doc_chunks):
        chunks.append(chunk)

print(f"\nCreated {len(chunks)} chunks")
print(f"Average chunk size: {sum(len(c) for c in chunks) / len(chunks):.0f} characters")

# Display sample chunk
print(f"\nSample chunk:")
print(f"{chunks[0]}")

## 5️⃣ Create Vector Index

Create OpenSearch index with vector field for semantic search.

### Index Configuration

- **Algorithm**: HNSW (Hierarchical Navigable Small World)
- **Distance Metric**: Cosine similarity
- **Engine**: FAISS (Facebook AI Similarity Search)
- **Parameters**:
  - `ef_construction: 512` - Build quality (higher = better but slower)
  - `m: 16` - Connections per node (higher = better recall)

In [ ]:
# Create index
success = opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    index_name=INDEX_NAME,
    force_recreate=True  # Delete if exists
)

if success:
    print("✓ Vector index created successfully")
else:
    print("❌ Failed to create index")

## 6️⃣ Generate Embeddings & Index Documents

Generate vector embeddings using Bedrock Titan and index into OpenSearch.

### Cost Tracking

**Titan Text Embeddings V2 Pricing:**
- $0.00002 per 1,000 input tokens
- ~750 words = 1,000 tokens

For this example with ~5 chunks:
- Estimated cost: < $0.001

In [ ]:
# Index documents with embeddings
print(f"Indexing {len(chunks)} chunks...")
start_time = time.time()

# Create metadata for each chunk
metadatas = [{"chunk_id": i, "source": "sample"} for i in range(len(chunks))]

# Index using RAG system
indexed_count = rag_system.index_documents(chunks, metadatas)

elapsed_time = time.time() - start_time

print(f"\n✓ Indexed {indexed_count} documents in {elapsed_time:.2f} seconds")
print(f"  Average: {elapsed_time/len(chunks):.2f} seconds per document")

# Verify indexing
doc_count = opensearch.get_document_count(INDEX_NAME)
print(f"\nVerification: {doc_count} documents in index")

## 7️⃣ Query the RAG System

Now let's test the RAG system with sample questions.

### Query Process

For each query:
1. **Embed**: Convert query to vector using Titan
2. **Search**: Find top-K similar documents in OpenSearch
3. **Context**: Format retrieved documents as context
4. **Generate**: Claude generates answer based on context

### Cost per Query

- Embedding: $0.00002 per query
- Claude Opus: ~$0.015 per 1K input tokens, ~$0.075 per 1K output
- Typical query cost: ~$0.05-0.10

In [ ]:
# Test questions
test_questions = [
    "What is AWS Bedrock?",
    "How does RAG work?",
    "What are vector embeddings?"
]

# Query each question
for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"Question {i}: {question}")
    print('='*70)
    
    # Measure latency
    start_time = time.time()
    
    # Query RAG system
    answer, sources = rag_system.query(
        question=question,
        top_k=TOP_K,
        return_sources=True
    )
    
    latency = (time.time() - start_time) * 1000
    
    # Display answer
    print(f"\n📝 Answer:")
    print(answer)
    
    # Display sources
    print(f"\n📚 Sources ({len(sources)} retrieved):")
    for j, source in enumerate(sources[:3], 1):  # Show top 3
        print(f"  {j}. [Score: {source['score']:.3f}] {source['text'][:100]}...")
    
    print(f"\n⏱️  Latency: {latency:.2f}ms")

## 8️⃣ Evaluation

Evaluate the RAG system's performance using metrics.

### Evaluation Metrics

- **Retrieval Metrics**: Precision, Recall, F1
- **Latency**: Time to generate answer
- **Answer Quality**: Using LLM-as-judge
- **Cost**: API call tracking

In [ ]:
# Initialize evaluator
evaluator = RAGEvaluator()

# Run evaluation on a test question
test_question = "What is AWS Bedrock?"
expected_keywords = ["foundation models", "API", "Anthropic", "Claude"]

print(f"Evaluating: {test_question}\n")

# Measure latency
answer, latency_ms = evaluator.measure_latency(
    rag_system.query,
    question=test_question,
    return_sources=False
)

print(f"Answer: {answer}\n")
print(f"Latency: {latency_ms:.2f}ms")

# Check if expected keywords are in answer
keywords_found = sum(1 for kw in expected_keywords if kw.lower() in answer.lower())
print(f"\nKeywords found: {keywords_found}/{len(expected_keywords)}")
for kw in expected_keywords:
    found = "✓" if kw.lower() in answer.lower() else "✗"
    print(f"  {found} {kw}")

## 9️⃣ Summary & Key Takeaways

### What We Built

✅ Complete Simple RAG system with AWS stack
✅ Document chunking and embedding generation
✅ Vector storage in OpenSearch Serverless
✅ Semantic retrieval and answer generation
✅ Evaluation and metrics tracking

### Performance Characteristics

- **Latency**: 1-3 seconds per query
- **Cost**: ~$0.05-0.10 per query
- **Scalability**: Handles thousands of documents
- **Accuracy**: Good for straightforward Q&A

### Limitations

1. **No reranking**: Top-K results may not be optimal
2. **Single query**: Doesn't explore query variations
3. **No graph structure**: Can't do multi-hop reasoning
4. **Fixed chunk size**: May split important context

### Next Steps

To improve beyond Simple RAG, consider:
- **Reranking**: Re-score retrieved documents
- **Fusion Retrieval**: Multiple query variations
- **Graph RAG**: Knowledge graph integration
- **HyDE**: Hypothetical document embeddings
- **Semantic Chunking**: Context-aware splitting

## 🧹 Cleanup (Optional)

Delete the index if you're done testing.

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later, run:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")